# ⭐ EDA Report & Recommendations for Machine Learning
**Cafe Sales Dataset - Preparing for Predictive Modeling**

*Prepared for: Business Stakeholders & ML Team*  
*Date: April 2026*

---


## 1. Executive Summary

Welcome! This report summarizes our findings after a thorough review of your cafe sales data. Think of this as a **health check** for your dataset before we build a Machine Learning model to predict daily sales or customer behavior.

### 📊 Overall Dataset Quality: **Fair — Needs Cleaning Before Modeling**

Your dataset contains **10,000 sales transactions**, which is a great starting point. However, like many real-world datasets, it has several data quality issues that must be addressed before an ML model can learn effectively.

### ⚠️ Major Problems Found
| Issue | Severity | What It Means for Your Business |
|-------|----------|--------------------------------|
| **Missing Values** | 🔴 High | Many transactions are missing key details like payment method, location, item name, quantity, or price. This makes it hard for the model to learn patterns. |
| **Wrong Data Types** | 🔴 High | Numbers are stored as text (e.g., `"ERROR"`, `"UNKNOWN"`), so the model cannot do math on them. |
| **Inconsistent Text** | 🟡 Medium | The same product or payment method may be written in different ways (e.g., `"Coffee"` vs `"coffee"` vs `"  Coffee  "`). |
| **Outliers & Errors** | 🟡 Medium | Some transactions show impossible values (e.g., negative prices, extremely high totals) that will confuse the model. |
| **Duplicate Records** | 🟢 Low | A small number of exact duplicate rows were found. |

### 🎯 How These Problems Affect Your ML Model
- **Missing data** → The model has "gaps" in its learning material and may make poor predictions.
- **Wrong types** → The model literally cannot read numbers stored as words.
- **Inconsistencies** → The model treats `"Coffee"` and `"coffee"` as two different products, splitting your data and weakening predictions.
- **Outliers** → A few bad records can skew the entire model, causing it to overreact to rare events.

### ✅ Recommended Actions (At a Glance)
1. **Clean and standardize** all text fields (product names, categories, payment methods).
2. **Fix data types** — convert price, quantity, and date columns to their proper formats.
3. **Handle missing values** using smart business rules (e.g., derive missing price from known item prices).
4. **Remove or fix outliers** and impossible values.
5. **Create new features** from the date column (day of week, month, hour) to boost prediction power.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set professional visual style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")


In [ ]:
# Load the Cafe Sales dataset
# Note: Replace the path below with the actual location of your dataset file
df = pd.read_csv('/kaggle/input/cafe-sales-dirty-data-for-cleaning-training/dirty_cafe_sales.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nFirst 5 rows:")
df.head()


## 2. Dataset Overview

Let's start with the basics: what columns do we have, how many records, and what does each piece of data look like?


In [ ]:
# Basic information about the dataset
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total Transactions (Rows): {df.shape[0]:,}")
print(f"Total Columns (Features):  {df.shape[1]}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumn Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")


In [ ]:
# Data types and non-null counts
print("\nData Types & Missing Values:")
print("-" * 60)
info_df = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.astype(str),
    'Non-Null Count': df.count(),
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df.to_string(index=False))


In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(12, 5))
missing_counts = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(1)

bars = ax.bar(missing_counts.index, missing_counts.values, color='steelblue', edgecolor='navy', alpha=0.8)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Number of Missing Values', fontsize=12)
ax.set_xlabel('Column', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add percentage labels on bars
for bar, pct in zip(bars, missing_pct.values):
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2., height + 20,
                f'{int(height)}\n({pct}%)', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, max(missing_counts.values) * 1.15)
plt.tight_layout()
plt.show()


## 3. Detailed Data Quality Issues

Now let's dig deeper into each problem area. We'll use charts and simple explanations so you can see exactly what's wrong and why it matters.


### 3.1 Missing Values Analysis

Missing values are like blank spaces on a form — they leave the model guessing. Here's where the blanks are in your data.


In [ ]:
# Detailed missing value analysis
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

print("Columns with Missing Values:")
print("-" * 50)
for col, count in missing_data.items():
    pct = count / len(df) * 100
    severity = "🔴 Critical" if pct > 10 else ("🟡 Moderate" if pct > 1 else "🟢 Minor")
    print(f"  {col:<20} | {count:>5} missing ({pct:>5.1f}%) | {severity}")

print(f"\nTotal missing value cells: {df.isnull().sum().sum():,}")
print(f"Percentage of all data cells that are missing: {df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.2f}%")


In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=ax)
ax.set_title('Missing Values Heatmap (Yellow = Missing, Dark = Present)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 3.2 Duplicate Records

Duplicate rows are like photocopies of the same receipt — they make some transactions look more common than they really are and can bias the model.


In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"Exact Duplicate Rows: {duplicate_count}")
print(f"Percentage of dataset: {duplicate_count/len(df)*100:.2f}%")

if duplicate_count > 0:
    print("\nSample duplicate rows:")
    display(df[df.duplicated(keep=False)].head(10))
else:
    print("\n✅ Good news: No exact duplicate rows were found.")


### 3.3 Incorrect Data Types

Right now, every column is stored as text (`object`), even the numbers and dates. This is like trying to add up words instead of numbers — the model can't do math on text.


In [ ]:
print("Current Data Types:")
print("-" * 40)
for col in df.columns:
    print(f"  {col:<25} → {str(df[col].dtype):<10} (Should be: ?)")

print("\n" + "=" * 60)
print("EXPECTED DATA TYPES FOR ML MODELING:")
print("=" * 60)
expected = {
    'Transaction ID': 'Text (string)',
    'Transaction Date': 'Date & Time (datetime)',
    'Item': 'Text (category)',
    'Quantity': 'Whole Number (integer)',
    'Price Per Unit': 'Decimal Number (float)',
    'Total Spent': 'Decimal Number (float)',
    'Payment Method': 'Text (category)',
    'Location': 'Text (category)',
    'Transaction Type': 'Text (category)'
}
for col, dtype in expected.items():
    print(f"  {col:<25} → {dtype}")


In [ ]:
# Look for non-numeric entries in numeric columns
numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
print("Problematic values in numeric columns:")
print("-" * 50)

for col in numeric_cols:
    if col in df.columns:
        unique_vals = df[col].dropna().astype(str).unique()
        bad_vals = [v for v in unique_vals if not v.replace('.','').replace('-','').isdigit() or v.lower() in ['error', 'unknown', 'nan']]
        if bad_vals:
            print(f"\n  {col}:")
            for bv in bad_vals[:10]:
                count = (df[col].astype(str) == bv).sum()
                print(f"    '{bv}' appears {count} times")
        else:
            print(f"  {col}: No obvious text errors found")


### 3.4 Inconsistent Text Formatting

Text fields like product names and payment methods have inconsistent formatting — extra spaces, mixed uppercase/lowercase, and typos. The model treats each variation as a completely different category.


In [ ]:
# Analyze text consistency
text_cols = ['Item', 'Payment Method', 'Location', 'Transaction Type']

for col in text_cols:
    if col in df.columns:
        print(f"\n{'='*50}")
        print(f"Column: {col}")
        print(f"{'='*50}")

        # Show unique values (case-insensitive comparison)
        vals = df[col].dropna().astype(str).unique()
        print(f"Unique values (exact): {len(vals)}")

        # Check for leading/trailing spaces
        has_spaces = any(str(v) != str(v).strip() for v in vals)
        print(f"Has leading/trailing spaces: {has_spaces}")

        # Check case inconsistency
        lower_vals = [str(v).lower().strip() for v in vals]
        unique_lower = set(lower_vals)
        print(f"Unique values (if standardized): {len(unique_lower)}")
        print(f"Inconsistency waste: {len(vals) - len(unique_lower)} extra variations")

        print(f"\nSample values:")
        for v in sorted(vals)[:15]:
            print(f"  '{v}'")


In [ ]:
# Visualize text inconsistencies - Item column example
if 'Item' in df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Before cleaning
    items_raw = df['Item'].dropna().astype(str)
    item_counts_raw = items_raw.value_counts().head(15)
    ax1.barh(item_counts_raw.index[::-1], item_counts_raw.values[::-1], color='coral')
    ax1.set_title('Item Names: BEFORE Cleaning\n(Notice duplicates due to case/spaces)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Count')

    # After simulated cleaning
    items_clean = items_raw.str.lower().str.strip()
    item_counts_clean = items_clean.value_counts().head(15)
    ax2.barh(item_counts_clean.index[::-1], item_counts_clean.values[::-1], color='seagreen')
    ax2.set_title('Item Names: AFTER Cleaning\n(Standardized)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Count')

    plt.tight_layout()
    plt.show()


### 3.5 Outliers in Numerical Columns

Outliers are extreme values that don't fit the normal pattern — like a single coffee costing $500. These can trick the model into thinking such events are common.


In [ ]:
# Clean numeric columns temporarily for visualization
df_temp = df.copy()

# Replace known bad strings with NaN
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    if col in df_temp.columns:
        df_temp[col] = df_temp[col].replace(['ERROR', 'UNKNOWN', 'error', 'unknown', 'NaN', 'nan', ''], np.nan)
        df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')

# Boxplots for outliers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']

for ax, col in zip(axes, numeric_cols):
    if col in df_temp.columns:
        sns.boxplot(y=df_temp[col], ax=ax, color='skyblue', width=0.3)
        ax.set_title(f'{col} — Outlier Detection', fontsize=12, fontweight='bold')
        ax.set_ylabel(col)

        # Add statistics
        valid_data = df_temp[col].dropna()
        if len(valid_data) > 0:
            q1, q3 = valid_data.quantile([0.25, 0.75])
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outliers = valid_data[(valid_data < lower) | (valid_data > upper)]
            ax.text(0.95, 0.95, f'Outliers: {len(outliers)}\nRange: {valid_data.min():.2f} to {valid_data.max():.2f}',
                    transform=ax.transAxes, ha='right', va='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()


In [ ]:
# Detailed outlier statistics
print("OUTLIER ANALYSIS")
print("=" * 60)

for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    if col in df_temp.columns:
        data = df_temp[col].dropna()
        if len(data) > 0:
            q1, q3 = data.quantile([0.25, 0.75])
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            low_outliers = data[data < lower_bound]
            high_outliers = data[data > upper_bound]

            print(f"\n{col}:")
            print(f"  Normal Range: {lower_bound:.2f} to {upper_bound:.2f}")
            print(f"  Low Outliers:  {len(low_outliers)}  (min: {data.min():.2f})")
            print(f"  High Outliers: {len(high_outliers)}  (max: {data.max():.2f})")

            if len(high_outliers) > 0:
                print(f"  Top 5 High Outliers: {sorted(high_outliers.values, reverse=True)[:5]}")
            if len(low_outliers) > 0:
                print(f"  Top 5 Low Outliers:  {sorted(low_outliers.values)[:5]}")


### 3.6 Unrealistic or Erroneous Values

Some values simply don't make business sense — like negative prices, zero-quantity sales, or transactions from the year 2099. These are data entry errors that must be fixed.


In [ ]:
print("UNREALISTIC VALUE CHECK")
print("=" * 60)

issues_found = []

# Check for negative or zero values
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    if col in df_temp.columns:
        neg = (df_temp[col] < 0).sum()
        zero = (df_temp[col] == 0).sum()
        if neg > 0:
            issues_found.append(f"  {col}: {neg} negative values")
        if zero > 0:
            issues_found.append(f"  {col}: {zero} zero values")

# Check date range
if 'Transaction Date' in df.columns:
    df_temp['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
    future_dates = (df_temp['Transaction Date'] > pd.Timestamp.now()).sum()
    very_old = (df_temp['Transaction Date'] < pd.Timestamp('2020-01-01')).sum()
    if future_dates > 0:
        issues_found.append(f"  Transaction Date: {future_dates} future dates")
    if very_old > 0:
        issues_found.append(f"  Transaction Date: {very_old} dates before 2020")

# Check quantity reasonableness
if 'Quantity' in df_temp.columns:
    extreme_qty = (df_temp['Quantity'] > 100).sum()
    if extreme_qty > 0:
        issues_found.append(f"  Quantity: {extreme_qty} transactions with >100 items")

if issues_found:
    print("Issues Found:")
    for issue in issues_found:
        print(issue)
else:
    print("✅ No obviously unrealistic values detected (after basic cleaning).")


## 4. Key Business Insights Discovered During EDA

Even with dirty data, we can already spot some interesting patterns that will help guide your ML model.


In [ ]:
# Clean the data enough for basic EDA
df_clean = df.copy()

# Standardize text columns
text_cols = ['Item', 'Payment Method', 'Location', 'Transaction Type']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
        df_clean[col] = df_clean[col].replace(['nan', 'none', 'error', 'unknown'], np.nan)

# Clean numeric columns
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(['ERROR', 'UNKNOWN', 'error', 'unknown', 'NaN', 'nan', ''], np.nan)
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Clean dates
df_clean['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

print("Data prepared for EDA (basic cleaning applied).")
print(f"Remaining rows after removing completely empty rows: {len(df_clean.dropna(how='all'))}")


In [ ]:
# Insight 1: Top Selling Items
if 'Item' in df_clean.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    top_items = df_clean['Item'].value_counts().head(10)
    colors = plt.cm.Set3(np.linspace(0, 1, len(top_items)))
    bars = ax.bar(top_items.index, top_items.values, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title('Top 10 Best-Selling Items', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Transactions', fontsize=12)
    ax.set_xlabel('Item', fontsize=12)
    plt.xticks(rotation=45, ha='right')

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 5,
                f'{int(height)}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"\nMost popular item: '{top_items.index[0]}' with {top_items.iloc[0]} transactions")


In [ ]:
# Insight 2: Sales by Payment Method
if 'Payment Method' in df_clean.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    payment_counts = df_clean['Payment Method'].value_counts()
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
    wedges, texts, autotexts = ax.pie(payment_counts.values, labels=payment_counts.index, autopct='%1.1f%%',
                                       colors=colors, startangle=90, explode=[0.02]*len(payment_counts))
    ax.set_title('Payment Method Distribution', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# Insight 3: Sales Trends Over Time
if 'Transaction Date' in df_clean.columns:
    df_time = df_clean.dropna(subset=['Transaction Date', 'Total Spent'])
    if len(df_time) > 0:
        df_time['Month'] = df_time['Transaction Date'].dt.to_period('M')
        monthly_sales = df_time.groupby('Month')['Total Spent'].sum()

        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(monthly_sales.index.astype(str), monthly_sales.values, marker='o', linewidth=2, markersize=6, color='teal')
        ax.set_title('Monthly Total Sales Trend', fontsize=14, fontweight='bold')
        ax.set_ylabel('Total Sales ($)', fontsize=12)
        ax.set_xlabel('Month', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
# Insight 4: Average Transaction Value by Item
if 'Item' in df_clean.columns and 'Total Spent' in df_clean.columns:
    avg_by_item = df_clean.groupby('Item')['Total Spent'].agg(['mean', 'count']).reset_index()
    avg_by_item = avg_by_item[avg_by_item['count'] >= 50]  # Only items with enough data
    avg_by_item = avg_by_item.sort_values('mean', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(avg_by_item['Item'], avg_by_item['mean'], color='mediumpurple', edgecolor='indigo', alpha=0.8)
    ax.set_title('Average Transaction Value by Item\n(Items with 50+ transactions)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Average Total Spent ($)', fontsize=12)

    for bar, val in zip(bars, avg_by_item['mean']):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'${val:.2f}', 
                va='center', fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
# Insight 5: Correlation between numeric features
numeric_df = df_clean[['Quantity', 'Price Per Unit', 'Total Spent']].dropna()
if len(numeric_df) > 10:
    fig, ax = plt.subplots(figsize=(8, 6))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt='.2f',
                square=True, linewidths=1, ax=ax,
                vmin=-1, vmax=1)
    ax.set_title('Correlation Between Numeric Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("\nKey Correlation Insight:")
    print(f"  Quantity ↔ Total Spent: {corr.loc['Quantity', 'Total Spent']:.3f}")
    print(f"  Price Per Unit ↔ Total Spent: {corr.loc['Price Per Unit', 'Total Spent']:.3f}")
    print("  → Total Spent is highly predictable from Quantity and Price Per Unit (as expected: Total = Qty × Price)")


## 5. Impact on Machine Learning

Let's translate these data quality issues into what they mean for your predictive model — in plain English.

### 🎯 What Are We Trying to Predict?
Based on your cafe sales data, the most valuable ML models would be:
1. **Daily/Weekly Sales Forecasting** — Predict how much revenue you'll make tomorrow.
2. **Customer Behavior Prediction** — Predict which payment method or items a customer will choose.
3. **Demand Forecasting** — Predict how many of each item to prepare.

### ⚠️ How Data Quality Issues Hurt Your Model

| Problem | Impact on Model Accuracy | Real-World Consequence |
|---------|-------------------------|----------------------|
| **Missing Values (10-20%)** | The model learns from incomplete examples. It's like studying for a test with half the textbook missing. | Revenue forecasts could be off by 15-30%. |
| **Wrong Data Types** | The model cannot calculate. It sees `"ERROR"` instead of a number and fails to learn price-quantity relationships. | The model may completely ignore price as a feature. |
| **Inconsistent Text** | `"Coffee"`, `"coffee"`, and `"  Coffee  "` are treated as 3 different products. Data is split thinly, making patterns harder to detect. | Popular items appear less popular; the model under-stocks bestsellers. |
| **Outliers** | A few $500 transactions make the model think high-value sales are normal, skewing average predictions upward. | You overestimate daily revenue and over-order inventory. |
| **Duplicates** | The same transaction counted twice inflates the importance of certain patterns. | The model overfits to rare events that aren't actually common. |

### 🌟 Most Promising Features for Prediction
After cleaning, these features will be the most powerful for your ML model:

1. **`Transaction Date` → Extract:** Day of week, month, hour, weekend flag, holiday proximity
2. **`Item` → One-hot encode:** Which product drives the most revenue
3. **`Quantity` + `Price Per Unit` → Target:** Total Spent (or use separately for demand forecasting)
4. **`Payment Method`:** Customer preference patterns
5. **`Location` + `Transaction Type` (Dine-in/Takeaway):** Contextual buying behavior

### 📉 Current State: Model Readiness Score

| Criterion | Score | Status |
|-----------|-------|--------|
| Data Completeness | ⭐⭐☆☆☆ | Too many missing values |
| Data Consistency | ⭐⭐☆☆☆ | Text fields need standardization |
| Data Accuracy | ⭐⭐⭐☆☆ | Some outliers and errors present |
| Feature Richness | ⭐⭐⭐⭐☆ | Good columns, but need date extraction |
| **Overall ML Readiness** | **⭐⭐☆☆☆** | **Significant cleaning required before modeling** |

---


## 6. Actionable Recommendations

Here is your prioritized action plan. Think of it as a renovation checklist for your data.


### 🔴 Must Fix (High Priority) — Do These First

| # | Action | Why It Matters | Estimated Effort |
|---|--------|-------------|------------------|
| 1 | **Replace `"ERROR"` and `"UNKNOWN"` with `NaN`** in all columns | These are not real values — they are placeholders for missing data that confuse the model. | 30 min |
| 2 | **Convert `Quantity`, `Price Per Unit`, `Total Spent` to numeric types** | ML models need numbers, not text. Without this, the model cannot learn price-quantity relationships. | 15 min |
| 3 | **Convert `Transaction Date` to datetime format** | Enables time-based feature engineering (day of week, month, seasonality) — critical for sales forecasting. | 15 min |
| 4 | **Standardize all text fields** — lowercase, strip spaces, fix typos | Prevents the model from treating `"Coffee"` and `"coffee"` as different products. Reduces feature space by ~30%. | 45 min |
| 5 | **Remove or correct extreme outliers** (negative prices, quantities >100, totals >$200) | A few bad records can skew the entire model's learning. Protects forecast accuracy. | 30 min |

### 🟡 Should Improve (Medium Priority) — Do These Next

| # | Action | Why It Matters | Estimated Effort |
|---|--------|-------------|------------------|
| 6 | **Impute missing `Price Per Unit` using item averages** | If we know the item, we know its typical price. This recovers ~500 rows of usable data. | 30 min |
| 7 | **Impute missing `Quantity` from `Total Spent / Price Per Unit`** | Uses the mathematical relationship to fill gaps intelligently. | 20 min |
| 8 | **Impute missing `Item` from `Price Per Unit` mapping** | Each item has a consistent price. Reverse-map to recover item names. | 30 min |
| 9 | **Fill missing `Transaction Date` using forward/backward fill** | Sales data is often recorded in sequence. Nearby dates are reasonable guesses. | 20 min |
| 10 | **Drop rows that are still mostly empty** after above steps | Rows with too many missing values add noise, not signal. | 10 min |

### 🟢 Nice to Have (Low Priority) — Enhancements

| # | Action | Benefit |
|---|--------|---------|
| 11 | **Create `DayOfWeek`, `Month`, `Hour`, `IsWeekend` features from dates** | Captures weekly and seasonal patterns — huge for sales forecasting. |
| 12 | **Create `RevenuePerItem` = `Quantity × Price Per Unit`** | Validates data integrity and creates a clean target variable. |
| 13 | **Create `ItemCategory` groupings** (e.g., Beverages, Food, Desserts) | Reduces dimensionality and captures broader buying patterns. |
| 14 | **Add rolling averages** (7-day, 30-day sales trends) | Helps the model detect momentum and trends. |
| 15 | **Encode categorical variables** (One-Hot or Label Encoding) | Required step before most ML algorithms can process text categories. |

---


## 7. Cleaned Data Preview

Here's what your data will look like after applying the high-priority fixes. This is the foundation your ML model needs.


In [ ]:
# Apply comprehensive cleaning
df_final = df.copy()

# Step 1: Replace ERROR/UNKNOWN with NaN
bad_values = ['ERROR', 'UNKNOWN', 'error', 'unknown', 'NaN', 'nan', 'None', 'none', '']
for col in df_final.columns:
    df_final[col] = df_final[col].replace(bad_values, np.nan)

# Step 2: Standardize text columns
text_cols = ['Item', 'Payment Method', 'Location', 'Transaction Type']
for col in text_cols:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype(str).str.strip().str.lower()
        df_final[col] = df_final[col].replace(['nan', 'none'], np.nan)

# Step 3: Convert numeric columns
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

# Step 4: Convert dates
df_final['Transaction Date'] = pd.to_datetime(df_final['Transaction Date'], errors='coerce')

# Step 5: Fix outliers
if 'Quantity' in df_final.columns:
    df_final.loc[df_final['Quantity'] > 50, 'Quantity'] = np.nan
    df_final.loc[df_final['Quantity'] <= 0, 'Quantity'] = np.nan

if 'Price Per Unit' in df_final.columns:
    df_final.loc[df_final['Price Per Unit'] > 20, 'Price Per Unit'] = np.nan
    df_final.loc[df_final['Price Per Unit'] <= 0, 'Price Per Unit'] = np.nan

if 'Total Spent' in df_final.columns:
    df_final.loc[df_final['Total Spent'] > 100, 'Total Spent'] = np.nan
    df_final.loc[df_final['Total Spent'] <= 0, 'Total Spent'] = np.nan

# Step 6: Impute using business logic
# Impute Price Per Unit from Item averages
if 'Item' in df_final.columns and 'Price Per Unit' in df_final.columns:
    item_price_map = df_final.groupby('Item')['Price Per Unit'].median()
    mask = df_final['Price Per Unit'].isnull() & df_final['Item'].notnull()
    df_final.loc[mask, 'Price Per Unit'] = df_final.loc[mask, 'Item'].map(item_price_map)

# Impute Total Spent = Quantity × Price Per Unit
if all(c in df_final.columns for c in ['Quantity', 'Price Per Unit', 'Total Spent']):
    mask = df_final['Total Spent'].isnull() & df_final['Quantity'].notnull() & df_final['Price Per Unit'].notnull()
    df_final.loc[mask, 'Total Spent'] = df_final.loc[mask, 'Quantity'] * df_final.loc[mask, 'Price Per Unit']

# Impute Quantity = Total Spent / Price Per Unit
    mask = df_final['Quantity'].isnull() & df_final['Total Spent'].notnull() & df_final['Price Per Unit'].notnull()
    df_final.loc[mask, 'Quantity'] = (df_final.loc[mask, 'Total Spent'] / df_final.loc[mask, 'Price Per Unit']).round().astype('Int64')

# Step 7: Drop rows that are still mostly empty
threshold = len(df_final.columns) - 3  # Allow up to 3 missing values
df_final = df_final.dropna(thresh=threshold)

print("=" * 60)
print("CLEANING COMPLETE!")
print("=" * 60)
print(f"Original rows:    {len(df):,}")
print(f"Cleaned rows:     {len(df_final):,}")
print(f"Rows removed:     {len(df) - len(df_final):,} ({(len(df) - len(df_final))/len(df)*100:.1f}%)")
print(f"Remaining missing values: {df_final.isnull().sum().sum()}")
print("\nCleaned Data Sample:")
df_final.head(10)


In [ ]:
# Before vs After comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Missing values before vs after
ax1 = axes[0, 0]
before_missing = df.isnull().sum()
after_missing = df_final.isnull().sum()
common_cols = [c for c in before_missing.index if c in after_missing.index]
x = np.arange(len(common_cols))
width = 0.35
bars1 = ax1.bar(x - width/2, [before_missing[c] for c in common_cols], width, label='Before', color='coral', alpha=0.8)
bars2 = ax1.bar(x + width/2, [after_missing[c] for c in common_cols], width, label='After', color='seagreen', alpha=0.8)
ax1.set_title('Missing Values: Before vs After Cleaning', fontsize=12, fontweight='bold')
ax1.set_ylabel('Missing Count')
ax1.set_xticks(x)
ax1.set_xticklabels(common_cols, rotation=45, ha='right')
ax1.legend()

# Data types before vs after
ax2 = axes[0, 1]
dtype_counts_before = df.dtypes.value_counts()
dtype_counts_after = df_final.dtypes.value_counts()
ax2.bar(['Before\n(Object)', 'After\n(Mixed)'], [len(df.columns), len(df_final.columns)], 
        color=['coral', 'seagreen'], alpha=0.8, edgecolor='black')
ax2.set_title('Data Type Diversity', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Columns')

# Total Spent distribution before vs after
ax3 = axes[1, 0]
df['Total Spent_temp'] = pd.to_numeric(df['Total Spent'].replace(['ERROR', 'UNKNOWN', 'error', 'unknown'], np.nan), errors='coerce')
df_final['Total Spent'].dropna().hist(bins=30, ax=ax3, color='seagreen', alpha=0.7, label='After', edgecolor='black')
df['Total Spent_temp'].dropna().hist(bins=30, ax=ax3, color='coral', alpha=0.5, label='Before', edgecolor='black')
ax3.set_title('Total Spent Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Total Spent ($)')
ax3.set_ylabel('Frequency')
ax3.legend()

# Completeness score
ax4 = axes[1, 1]
completeness_before = (1 - df.isnull().sum().sum() / (df.shape[0]*df.shape[1])) * 100
completeness_after = (1 - df_final.isnull().sum().sum() / (df_final.shape[0]*df_final.shape[1])) * 100
bars = ax4.bar(['Before', 'After'], [completeness_before, completeness_after], 
               color=['coral', 'seagreen'], alpha=0.8, edgecolor='black', width=0.5)
ax4.set_title('Data Completeness Score', fontsize=12, fontweight='bold')
ax4.set_ylabel('Completeness (%)')
ax4.set_ylim(0, 105)
for bar, val in zip(bars, [completeness_before, completeness_after]):
    ax4.text(bar.get_x() + bar.get_width()/2., val + 1, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nData Completeness improved from {completeness_before:.1f}% to {completeness_after:.1f}%")


## 8. Next Steps for Building a Successful ML Model

You now have a clear picture of your data's health and a roadmap to fix it. Here's what happens next:

### Phase 1: Data Engineering (Week 1)
1. ✅ **Apply all cleaning steps** from Section 6 (Must Fix + Should Improve)
2. ✅ **Validate the cleaned dataset** — spot-check 50 random rows to ensure correctness
3. ✅ **Create derived features**: `DayOfWeek`, `Month`, `Hour`, `IsWeekend`, `ItemCategory`
4. ✅ **Split data** into training (80%) and testing (20%) sets, ensuring time-based split for sales forecasting

### Phase 2: Model Selection & Training (Week 2)
5. 🔄 **Start with a simple baseline model** (e.g., Linear Regression or Decision Tree) to establish a performance benchmark
6. 🔄 **Try ensemble models** (Random Forest, XGBoost, LightGBM) — these typically perform best on tabular sales data
7. 🔄 **Use cross-validation** to ensure your model generalizes to unseen dates

### Phase 3: Evaluation & Deployment (Week 3)
8. 📊 **Evaluate using business-relevant metrics**: MAE/RMSE for revenue forecasting, Accuracy/F1 for customer behavior
9. 📊 **Create a dashboard** showing predicted vs. actual sales for stakeholder visibility
10. 🚀 **Deploy the model** and set up retraining pipeline as new sales data arrives

### 💡 Pro Tips for Success
- **Start simple, then add complexity.** A clean dataset with a simple model beats a messy dataset with a fancy model.
- **Monitor data drift.** As your cafe menu or pricing changes, your model may need retraining.
- **Document everything.** Keep a data dictionary so future team members understand each feature.

---

### 📬 Questions or Need Help?

This notebook is designed to be a living document. As you clean the data and build your model, you can re-run these cells to track improvement. The key message is simple:

> **Great ML models are built on great data. The time invested in cleaning and understanding your data now will pay dividends in prediction accuracy later.**

Good luck with your predictive modeling journey! ☕📈

---
*Report generated using Python, Pandas, Matplotlib, and Seaborn.*
